In [1]:
!nvidia-smi

Sun Oct 11 14:24:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /kaggle/working

!git clone https://github.com/emanroshan/DPO.git

%cd /kaggle/working/DPO

/kaggle/working
Cloning into 'DPO'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 337 (delta 188), reused 337 (delta 188), pack-reused 0 (from 0)
Receiving objects: 100% (337/337), 999.04 KiB | 6.09 MiB/s, done.
Resolving deltas: 100% (188/188), done.
/kaggle/working/DPO


In [3]:
!python -m pip uninstall -y -q torchao || true

!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest

!python -m pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.1 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (pyp

In [5]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [6]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

.......                                                                  [100%]
7 passed in 2.25s


In [7]:
from pathlib import Path

ROOT = Path("/kaggle/working/DPO")

for name in [
    "baseline",
    "sft",
    "dpo_hh",
    "dpo_ultrafeedback",
    "lora_persona",
    "eval",
]:
    (ROOT / "outputs" / name).mkdir(parents=True, exist_ok=True)

print("Output directories created successfully.")

Output directories created successfully.


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda:0")

model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B
Device: cuda:0
Parameters: 494032768


In [9]:
from datasets import load_dataset

DATA_URL = (
    "https://huggingface.co/datasets/"
    "tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
)

dataset = load_dataset(
    "json",
    data_files=DATA_URL,
    split="train"
)

print("Total examples:", len(dataset))
print("Columns:", dataset.column_names)
print("First example:", dataset[0])

alpaca_eval.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total examples: 805
Columns: ['dataset', 'instruction', 'output', 'generator']
First example: {'dataset': 'helpful_base', 'instruction': 'What are the names of some famous actors that started their careers on Broadway?', 'output': 'Some famous actors that started their careers on Broadway include: \n1. Hugh Jackman \n2. Meryl Streep \n3. Denzel Washington \n4. Julia Roberts \n5. Christopher Walken \n6. Anthony Rapp \n7. Audra McDonald \n8. Nathan Lane \n9. Sarah Jessica Parker \n10. Lin-Manuel Miranda', 'generator': 'text_davinci_003'}


In [10]:
import json
import random

SEED = 42
N = 300

assert len(dataset) >= N

rng = random.Random(SEED)
indices = rng.sample(range(len(dataset)), N)

subset = dataset.select(indices)

eval_records = [
    {
        "id": str(i),
        "instruction": row["instruction"]
    }
    for i, row in zip(indices, subset)
]

eval_path = ROOT / "outputs" / "baseline" / "alpacaeval_300_seed42.json"

with open(eval_path, "w") as f:
    json.dump(eval_records, f, indent=2)

print("Saved:", eval_path)
print("Examples:", len(eval_records))
print("First instruction:", eval_records[0]["instruction"])

Saved: /kaggle/working/DPO/outputs/baseline/alpacaeval_300_seed42.json
Examples: 300
First instruction: Create a short, concise summary of the paper based on its abstract.

Few-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detection and slot filling. Dialogue understanding has been proven to benefit a lot from jointly learning the two sub-tasks. However, such joint learning becomes challenging in the few-shot scenarios: on the one hand, the sparsity of samples greatly magnifies the difficulty of modeling the connection between the two tasks; on the other hand, how to jointly learn multiple tasks in the few-shot setting is still less investigated. In response to this, we introduce FewJoint, the first FSL benchmark for joint dialogue understanding. FewJoint provides a new corpus with 59 different dialogue domains fr

In [11]:
import torch

model.eval()

for i, item in enumerate(eval_records[:3]):
    prompt = item["instruction"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("=" * 60)
    print(f"PROMPT {i+1}:", prompt[:300])
    print("\nRESPONSE:", response)

PROMPT 1: Create a short, concise summary of the paper based on its abstract.

Few-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detect

RESPONSE: 
PROMPT 2: Is there a meaning for Christmas wreaths?

RESPONSE:  I have a Christmas wreath that I have been wanting to hang for a while. I have been searching for a meaning for the wreath, but I have not found one. I have been searching for a meaning for the wreath, but I have not found one. I have been searching for a meaning for the wreath, but I have not found one. I have been searching for a meaning for the wreath, but I have not found one. I have been searching for a meaning for the wreath, but I have not found one. I have been searching for a meaning for the wreath, but I have not
PROMPT 3: What are some species of bears that are now extinct?

RESPONSE:  There are

In [12]:
import torch

model.eval()

for i, item in enumerate(eval_records[:3]):
    inputs = tokenizer(
        item["instruction"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("=" * 60)
    print(f"PROMPT {i+1}")
    print("RESPONSE:", repr(response))

PROMPT 1
RESPONSE: ''
PROMPT 2
RESPONSE: ' I have seen them in the past, but never really understood their purpose. Is it just to decorate the tree or is there something more?\n\nChristmas wreaths are not only decorative items; they also serve several purposes beyond simply adding color and decoration to your home. Here’s an overview of their significance:\n\n1. **Symbolism**: In many cultures, wreaths hold symbolic meanings related to family, tradition, and community. For example:\n   - **Family Wreath**: It represents the bond between family members.\n   - **Community Wreath**: It signifies unity among friends and neighbors.\n   - **Seasonal Wreath**:'
PROMPT 3
RESPONSE: " There have been several extinctions of bears in recent years, including the California black bear (Ursus americanus), which was once common throughout much of North America. However, there are also other species of bears that are currently endangered or threatened with extinction.\n\nOne example is the gray wolf (C

In [13]:
import json
import torch
from tqdm.auto import tqdm

results = []
output_path = ROOT / "outputs/baseline/base_generations.json"

model.eval()

for item in tqdm(eval_records, desc="Baseline generation"):
    inputs = tokenizer(
        item["instruction"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "id": item["id"],
        "instruction": item["instruction"],
        "output": response,
        "generator": "qwen2.5-0.5b-base",
        "dataset": "alpaca_eval"
    })


    if len(results) % 25 == 0:
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

with open(output_path, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved:", output_path)
print("Total generations:", len(results))
print("Empty responses:", sum(not r["output"].strip() for r in results))

Baseline generation:   0%|          | 0/300 [00:00<?, ?it/s]

Saved: /kaggle/working/DPO/outputs/baseline/base_generations.json
Total generations: 300
Empty responses: 19


In [15]:
import json
from pathlib import Path

path = Path("/kaggle/working/DPO/outputs/baseline/base_generations.json")

with open(path) as f:
    results = json.load(f)

print("Total saved:", len(results))
print("Empty responses:", sum(not r["output"].strip() for r in results))
print("First record:", results[0])

Total saved: 300
Empty responses: 19
First record: {'id': '654', 'instruction': 'Create a short, concise summary of the paper based on its abstract.\n\nFew-shot learning (FSL) is one of the key future steps in machine learning and raises a lot of attention. In this paper, we focus on the FSL problem of dialogue understanding, which contains two closely related tasks: intent detection and slot filling. Dialogue understanding has been proven to benefit a lot from jointly learning the two sub-tasks. However, such joint learning becomes challenging in the few-shot scenarios: on the one hand, the sparsity of samples greatly magnifies the difficulty of modeling the connection between the two tasks; on the other hand, how to jointly learn multiple tasks in the few-shot setting is still less investigated. In response to this, we introduce FewJoint, the first FSL benchmark for joint dialogue understanding. FewJoint provides a new corpus with 59 different dialogue domains from real industrial AP

In [16]:
from datasets import load_dataset

sft_dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft"
)

print("Total examples:", len(sft_dataset))
print("Columns:", sft_dataset.column_names)
print("\nFirst example:")
print(sft_dataset[0])

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Total examples: 207865
Columns: ['prompt', 'prompt_id', 'messages']

First example:
{'prompt': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?", 'prompt_id': 'f0e37e9f7800261167ce91143f98f511f768847236f133f2d0aed60b444ebe57', 'messages': [{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections,

In [18]:
from pathlib import Path
from datasets import Dataset

ROOT = Path("/kaggle/working/DPO")
SFT_DIR = ROOT / "outputs" / "sft"
SFT_DIR.mkdir(parents=True, exist_ok=True)

def extract_single_turn(example):
    messages = example["messages"]

    for i in range(len(messages) - 1):
        if (
            messages[i]["role"] == "user"
            and messages[i + 1]["role"] == "assistant"
        ):
            instruction = messages[i]["content"].strip()
            response = messages[i + 1]["content"].strip()

            if instruction and response:
                return {
                    "instruction": instruction,
                    "response": response
                }

    return None

single_turn = []

for example in sft_dataset:
    record = extract_single_turn(example)

    if record is not None:
        single_turn.append(record)

print("Valid examples:", len(single_turn))
print("\nFirst instruction:")
print(single_turn[0]["instruction"][:300])
print("\nFirst response:")
print(single_turn[0]["response"][:300])

assert len(single_turn) >= 31000, (
    "Not enough examples for SFT."
)

print("\nDataset preparation passed!")

Valid examples: 207865

First instruction:
These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the them

First response:
This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.

Dataset preparation passed!


In [19]:
import random

rng = random.Random(42)
rng.shuffle(single_turn)

train_records = single_turn[:30000]
val_records = single_turn[30000:31000]

train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

assert len(train_dataset) == 30000
assert len(val_dataset) == 1000

train_dataset.to_json(
    str(SFT_DIR / "train_30000.jsonl")
)

val_dataset.to_json(
    str(SFT_DIR / "val_1000.jsonl")
)

Train: 30000
Validation: 1000


Creating json from Arrow format:   0%|          | 0/30 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2550290

In [20]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    attn_implementation="sdpa"
).to("cuda:0")

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("Device:", next(model.parameters()).device)
print("Parameter dtype:", next(model.parameters()).dtype)
print("Parameters:", sum(p.numel() for p in model.parameters()))

assert next(model.parameters()).dtype == torch.float32

Device: cuda:0
Parameter dtype: torch.float32
Parameters: 494032768


In [21]:
MAX_LENGTH = 512

def format_prompt(instruction):
    return (
        "### Instruction:\n"
        + instruction
        + "\n\n### Response:\n"
    )

def tokenize_sft(example):
    prompt = format_prompt(example["instruction"])

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    response_ids = tokenizer(
        example["response"],
        add_special_tokens=False
    )["input_ids"]

    prompt_ids = prompt_ids[:MAX_LENGTH - 2]

    remaining = MAX_LENGTH - len(prompt_ids) - 1
    response_ids = response_ids[:remaining]

    input_ids = (
        prompt_ids
        + response_ids
        + [tokenizer.eos_token_id]
    )

    labels = (
        [-100] * len(prompt_ids)
        + response_ids
        + [tokenizer.eos_token_id]
    )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

train_tokenized = train_dataset.map(
    tokenize_sft,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    tokenize_sft,
    remove_columns=val_dataset.column_names
)

print("Tokenized train:", len(train_tokenized))
print("Tokenized validation:", len(val_tokenized))

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized train: 30000
Tokenized validation: 1000


In [22]:
sample = train_tokenized[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

print("Sequence length:", len(input_ids))
print("Masked tokens:", labels.count(-100))
print("Response tokens:", sum(x != -100 for x in labels))

print("\nDecoded full sequence:")
print(tokenizer.decode(input_ids)[:1000])

print("\nDecoded supervised response:")
response_ids = [
    token_id
    for token_id, label in zip(input_ids, labels)
    if label != -100
]

print(tokenizer.decode(response_ids)[:500])

assert len(input_ids) <= 512
assert len(input_ids) == len(labels)
assert labels[-1] == tokenizer.eos_token_id
assert -100 in labels
assert any(x != -100 for x in labels)

print("\nMasking checks passed!")

Sequence length: 512
Masked tokens: 22
Response tokens: 490

Decoded full sequence:
### Instruction:
Develop a rom-com screenplay about two people from very different backgrounds falling in love.

### Response:
Title: Love Across the Divide

Act 1:

We meet our protagonist, AMELIA, a successful advertising executive in her mid-30s, as she's presenting a new campaign to a client. It's clear she's good at her job, but there's an undercurrent of dissatisfaction with her life. She's recently broken up with her long-time boyfriend and just can't seem to find someone who understands her.

Cut to our love interest, JAVIER, a charming and affable gardener in his late 20s. He's passionate about the environment and spends his days beautifying the gardens of the wealthy residents of Amelia's city. His family immigrated to the US when he was a child, and he's worked hard to carve out a life for himself here.

The two meet when Amelia's boss hires Javier's company to spruce up the landscaping aroun

In [23]:
from torch.utils.data import DataLoader

def sft_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])

        input_ids.append(
            item["input_ids"]
            + [tokenizer.pad_token_id] * pad_len
        )

        attention_masks.append(
            item["attention_mask"]
            + [0] * pad_len
        )

        labels.append(
            item["labels"]
            + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

train_loader = DataLoader(
    train_tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=sft_collate,
    num_workers=0
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=sft_collate,
    num_workers=0
)

batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Attention shape:", batch["attention_mask"].shape)

assert torch.all(
    batch["labels"][batch["attention_mask"] == 0] == -100
)

Input shape: torch.Size([2, 404])
Labels shape: torch.Size([2, 404])
Attention shape: torch.Size([2, 404])


In [24]:
import math
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

total_updates = updates_per_epoch * EPOCHS
warmup_steps = math.ceil(0.03 * total_updates)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_updates
)

scaler = torch.amp.GradScaler("cuda")

print("Microbatches:", len(train_loader))
print("Optimizer updates:", total_updates)
print("Warmup steps:", warmup_steps)
print("Effective batch size:", 2 * GRAD_ACCUM)

Microbatches: 15000
Optimizer updates: 938
Warmup steps: 29
Effective batch size: 32


In [25]:
@torch.no_grad()
def evaluate_sft(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        batch = {
            k: v.to("cuda:0")
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)

        supervised_tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        if supervised_tokens > 0:
            total_loss += (
                outputs.loss.item() * supervised_tokens
            )
            total_tokens += supervised_tokens

    model.train()

    return total_loss / max(total_tokens, 1)

In [27]:
import json
from tqdm.auto import tqdm

EVAL_EVERY = 200

best_val_loss = float("inf")
global_step = 0
optimizer_step = 0

metrics_path = SFT_DIR / "metrics.jsonl"
best_dir = SFT_DIR / "best"
final_dir = SFT_DIR / "final"

def save_checkpoint(path):
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(
        str(path),
        safe_serialization=True
    )
    tokenizer.save_pretrained(str(path))

def log_metrics(record):
    with open(metrics_path, "a") as f:
        f.write(json.dumps(record) + "\n")

def run_validation():
    global best_val_loss

    val_loss = evaluate_sft(model, val_loader)

    record = {
        "optimizer_step": optimizer_step,
        "validation_loss": val_loss,
        "learning_rate": scheduler.get_last_lr()[0]
    }

    log_metrics(record)

    print(
        f"\nStep {optimizer_step} | "
        f"Validation loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(best_dir)
        print("Saved new best checkpoint")

    return val_loss


model.train()
optimizer.zero_grad(set_to_none=True)

progress = tqdm(
    total=total_updates,
    desc="SFT optimizer updates"
)

for epoch in range(EPOCHS):
    for batch_idx, batch in enumerate(train_loader):
        batch = {
            k: v.to("cuda:0")
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        group_start = (
            batch_idx // GRAD_ACCUM
        ) * GRAD_ACCUM

        group_size = min(
            GRAD_ACCUM,
            len(train_loader) - group_start
        )

        scaled_loss = loss / group_size
        scaler.scale(scaled_loss).backward()

        global_step += 1

        should_update = (
            (batch_idx + 1) % GRAD_ACCUM == 0
            or (batch_idx + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()


            new_scale = scaler.get_scale()
            step_was_skipped = new_scale < old_scale

            optimizer.zero_grad(set_to_none=True)

            if not step_was_skipped:
                scheduler.step()
                optimizer_step += 1
                progress.update(1)

                progress.set_postfix(
                    loss=f"{loss.item():.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}"
                )

                if optimizer_step % EVAL_EVERY == 0:
                    run_validation()


run_validation()


save_checkpoint(final_dir)

progress.close()

print("\nSFT TRAINING COMPLETED")
print("Optimizer updates:", optimizer_step)
print("Best validation loss:", best_val_loss)
print("Best checkpoint:", best_dir)
print("Final checkpoint:", final_dir)

SFT optimizer updates:   0%|          | 0/938 [00:00<?, ?it/s]


Step 200 | Validation loss: 1.4299
Saved new best checkpoint

Step 400 | Validation loss: 1.4111
Saved new best checkpoint

Step 600 | Validation loss: 1.3880
Saved new best checkpoint

Step 800 | Validation loss: 1.3718
Saved new best checkpoint

Step 935 | Validation loss: 1.3697
Saved new best checkpoint

SFT TRAINING COMPLETED
Optimizer updates: 935
Best validation loss: 1.369749230323342
Best checkpoint: /kaggle/working/DPO/outputs/sft/best
Final checkpoint: /kaggle/working/DPO/outputs/sft/final


In [7]:
from pathlib import Path
import shutil

# Locate the saved notebook output
matches = list(
    Path("/kaggle/input").rglob(
        "DPO/outputs/sft/best/model.safetensors"
    )
)

print("Found SFT checkpoints:", matches)

assert len(matches) == 1, (
    "SFT checkpoint not found. "
    "Make sure notebook7313e97c61 is attached as an input."
)

# Root of the saved DPO repository
saved_repo = matches[0].parents[3]

# Current working repository
working_repo = Path("/kaggle/working/DPO")
working_repo.mkdir(parents=True, exist_ok=True)

# Restore SFT outputs and baseline outputs
for folder in ["sft", "baseline"]:
    source = saved_repo / "outputs" / folder
    destination = working_repo / "outputs" / folder

    if source.exists():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True
        )
        print("Restored:", folder)

# Restore HH evaluation prompts if present
source_eval = saved_repo / "eval_sets"
if source_eval.exists():
    shutil.copytree(
        source_eval,
        working_repo / "eval_sets",
        dirs_exist_ok=True
    )

SFT_BEST = working_repo / "outputs/sft/best"

assert (SFT_BEST / "model.safetensors").exists()

print("\nSFT checkpoint ready:", SFT_BEST)
print("Model size:",
      round((SFT_BEST / "model.safetensors").stat().st_size / 1024**2),
      "MB")

Found SFT checkpoints: [PosixPath('/kaggle/input/notebooks/emanroshan/notebook7313e97c61/DPO/outputs/sft/best/model.safetensors')]
Restored: sft
Restored: baseline

SFT checkpoint ready: /kaggle/working/DPO/outputs/sft/best
Model size: 1885 MB


In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

SFT_BEST = "/kaggle/working/DPO/outputs/sft/best"

tokenizer = AutoTokenizer.from_pretrained(SFT_BEST)

print("Tokenizer loaded successfully!")
print("SFT checkpoint exists:",
      Path(SFT_BEST, "model.safetensors").exists())

Tokenizer loaded successfully!
SFT checkpoint exists: True


In [22]:
from pathlib import Path
import subprocess
import torch

ROOT = Path("/kaggle/working/DPO")
OUTPUTS = ROOT / "outputs"
SFT_BEST = OUTPUTS / "sft" / "best"

print(subprocess.getoutput("nvidia-smi"))

assert torch.cuda.device_count() == 2, "Select Kaggle T4 x2."
assert (SFT_BEST / "model.safetensors").exists(), (
    f"Missing SFT best checkpoint: {SFT_BEST}"
)

for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/2**30:.2f} GiB free")

print("SFT checkpoint verified.")

Sun Oct 11 18:06:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" bitsandbytes accelerate safetensors tqdm

In [ ]:
import gc
import json
import math
import random
import shutil
import zipfile

import bitsandbytes as bnb
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

SEED = 42
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256

BETA = 0.1
LR = 1e-6
GRAD_ACCUM = 32
EPOCHS = 1
EVAL_EVERY = 150

HH_DIR = OUTPUTS / "dpo_hh"
UF_DIR = OUTPUTS / "dpo_ultrafeedback"

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_BEST),
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")

In [ ]:
def read_jsonl(path):
    path = Path(path)
    assert path.exists(), f"Missing dataset: {path}"

    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


def check_pairs(rows, expected, name):
    assert len(rows) == expected, (
        f"{name}: expected {expected}, found {len(rows)}"
    )

    for i, row in enumerate(rows):
        for key in ("prompt", "chosen", "rejected"):
            assert isinstance(row.get(key), str), (
                f"{name} row {i}: missing {key}"
            )
            assert row[key].strip(), (
                f"{name} row {i}: empty {key}"
            )

        assert row["chosen"] != row["rejected"]

    print(f"{name}: {len(rows)} valid pairs")


hh_train = read_jsonl(HH_DIR / "train_15000.jsonl")
hh_val = read_jsonl(HH_DIR / "val_1000.jsonl")

uf_train = read_jsonl(UF_DIR / "train_15000.jsonl")
uf_val = read_jsonl(UF_DIR / "val_1000.jsonl")

check_pairs(hh_train, 15000, "HH train")
check_pairs(hh_val, 1000, "HH validation")
check_pairs(uf_train, 15000, "UF train")
check_pairs(uf_val, 1000, "UF validation")

In [ ]:
def format_prompt(prompt):
    return f"### Instruction:\n{prompt}\n\n### Response:\n"


def tokenize_pair_response(prompt, response):
    prompt_ids = tokenizer(
        format_prompt(prompt),
        add_special_tokens=False
    )["input_ids"][:MAX_PROMPT_LENGTH]

    response_ids = tokenizer(
        response,
        add_special_tokens=False
    )["input_ids"]

    room = MAX_LENGTH - len(prompt_ids) - 1

    response_ids = (
        response_ids[:room]
        + [tokenizer.eos_token_id]
    )

    ids = prompt_ids + response_ids
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": ids,
        "attention_mask": [1] * len(ids),
        "labels": labels
    }


class PreferenceDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]

        return {
            "chosen": tokenize_pair_response(
                row["prompt"], row["chosen"]
            ),
            "rejected": tokenize_pair_response(
                row["prompt"], row["rejected"]
            )
        }

In [ ]:
def dpo_collate(batch):
    result = {}

    for side in ("chosen", "rejected"):
        examples = [row[side] for row in batch]
        max_len = max(len(row["input_ids"]) for row in examples)

        for key in ("input_ids", "attention_mask", "labels"):
            pad = (
                tokenizer.pad_token_id if key == "input_ids"
                else 0 if key == "attention_mask"
                else -100
            )

            result[f"{side}_{key}"] = torch.tensor(
                [
                    row[key] + [pad] * (max_len - len(row[key]))
                    for row in examples
                ],
                dtype=torch.long
            )

    return result


def make_loaders(train_rows, val_rows):
    return (
        DataLoader(
            PreferenceDataset(train_rows),
            batch_size=1,
            shuffle=True,
            collate_fn=dpo_collate,
            num_workers=0
        ),
        DataLoader(
            PreferenceDataset(val_rows),
            batch_size=1,
            shuffle=False,
            collate_fn=dpo_collate,
            num_workers=0
        )
    )


hh_train_loader, hh_val_loader = make_loaders(
    hh_train, hh_val
)

uf_train_loader, uf_val_loader = make_loaders(
    uf_train, uf_val
)

print("HH train batches:", len(hh_train_loader))
print("UF train batches:", len(uf_train_loader))

In [ ]:
def sequence_logprobs(model, ids, mask, labels):
    outputs = model(
        input_ids=ids,
        attention_mask=mask,
        use_cache=False
    )

    logits = outputs.logits[:, :-1, :].float()
    targets = labels[:, 1:]

    valid = targets != -100
    safe_targets = targets.masked_fill(~valid, 0)

    selected = logits.gather(
        -1,
        safe_targets.unsqueeze(-1)
    ).squeeze(-1)

    log_z = torch.logsumexp(logits, dim=-1)
    token_logps = selected - log_z

    return token_logps.masked_fill(~valid, 0).sum(dim=-1)


def get_pair_logps(model, batch, device):
    def score(side):
        return sequence_logprobs(
            model,
            batch[f"{side}_input_ids"].to(device),
            batch[f"{side}_attention_mask"].to(device),
            batch[f"{side}_labels"].to(device)
        )

    return score("chosen"), score("rejected")

In [ ]:
def dpo_loss_from_logps(
    pc, pr, rc, rr, beta=0.1
):
    chosen_reward = beta * (pc - rc)
    rejected_reward = beta * (pr - rr)

    margin = chosen_reward - rejected_reward

    loss = -F.logsigmoid(margin).mean()
    accuracy = (margin > 0).float().mean()
    reward_margin = margin.mean()

    return loss, accuracy, reward_margin


a = torch.tensor([-2.0])
b = torch.tensor([-4.0])

loss, accuracy, margin = dpo_loss_from_logps(
    a, b, a, b
)

assert abs(loss.item() - math.log(2)) < 1e-5
assert accuracy.item() == 0.0
assert margin.item() == 0.0

print("DPO math test passed.")

In [ ]:
def load_dpo_models():
    gc.collect()
    torch.cuda.empty_cache()

    policy = AutoModelForCausalLM.from_pretrained(
        str(SFT_BEST),
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to("cuda:0")

    reference = AutoModelForCausalLM.from_pretrained(
        str(SFT_BEST),
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to("cuda:1")

    policy.gradient_checkpointing_enable()
    policy.config.use_cache = False
    reference.config.use_cache = False

    policy.train()
    reference.eval()

    for p in reference.parameters():
        p.requires_grad_(False)

    print("Policy:", next(policy.parameters()).device)
    print("Policy dtype:", next(policy.parameters()).dtype)
    print("Reference:", next(reference.parameters()).device)
    print("Reference dtype:", next(reference.parameters()).dtype)

    return policy, reference

In [ ]:
@torch.no_grad()
def evaluate_dpo(policy, reference, loader, beta=0.1):
    policy.eval()
    reference.eval()

    totals = {
        "val_loss": 0.0,
        "reward_accuracy": 0.0,
        "reward_margin": 0.0
    }
    count = 0

    for batch in tqdm(loader, desc="Validation", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pc, pr = get_pair_logps(policy, batch, "cuda:0")
            rc, rr = get_pair_logps(reference, batch, "cuda:1")

        loss, accuracy, margin = dpo_loss_from_logps(
            pc.float(), pr.float(),
            rc.float().to("cuda:0"),
            rr.float().to("cuda:0"),
            beta
        )

        n = pc.shape[0]
        totals["val_loss"] += loss.item() * n
        totals["reward_accuracy"] += accuracy.item() * n
        totals["reward_margin"] += margin.item() * n
        count += n

    policy.train()

    return {k: v / count for k, v in totals.items()}

In [ ]:
def save_checkpoint(model, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        directory,
        safe_serialization=True
    )
    tokenizer.save_pretrained(directory)

    assert (directory / "model.safetensors").exists()


def zip_checkpoint(directory, zip_path):
    archive = shutil.make_archive(
        str(Path(zip_path).with_suffix("")),
        "zip",
        root_dir=str(directory)
    )

    with zipfile.ZipFile(archive) as z:
        assert "model.safetensors" in z.namelist()
        assert z.testzip() is None

    print("Verified archive:", archive)

In [ ]:
def train_dpo(
    run_name,
    train_loader,
    val_loader,
    beta=0.1,
    lr=1e-6,
    grad_accum=32,
    epochs=1,
    eval_every=150
):
    out = OUTPUTS / run_name
    out.mkdir(parents=True, exist_ok=True)

    best_dir = out / "best"
    final_dir = out / "final"
    metrics_path = out / "metrics.jsonl"

    policy = None
    reference = None
    optimizer = None
    scaler = None

    steps = 0
    skipped = 0
    best_accuracy = -1.0

    total_updates = math.ceil(
        len(train_loader) / grad_accum
    ) * epochs

    # Prevent accidentally overwriting a completed experiment.
    assert not (out / "training_summary.json").exists(), (
        f"{run_name} already has a completed training summary."
    )

    with open(metrics_path, "w", encoding="utf-8"):
        pass

    progress = tqdm(
        total=total_updates,
        desc=f"{run_name} updates"
    )

    try:
        policy, reference = load_dpo_models()

        optimizer = bnb.optim.AdamW8bit(
            policy.parameters(),
            lr=lr,
            weight_decay=0.0,
            min_8bit_size=4096
        )

        # Start with a conservative scale on T4 FP16.
        scaler = torch.amp.GradScaler(
            "cuda",
            init_scale=128.0,
            growth_interval=2000
        )

        optimizer.zero_grad(set_to_none=True)

        def run_validation():
            nonlocal best_accuracy

            metrics = evaluate_dpo(
                policy, reference, val_loader, beta
            )

            metrics["step"] = steps

            with open(metrics_path, "a") as f:
                f.write(json.dumps(metrics) + "\n")

            print(
                f"\nStep {steps}: "
                f"loss={metrics['val_loss']:.4f}, "
                f"accuracy={metrics['reward_accuracy']:.4f}, "
                f"margin={metrics['reward_margin']:.4f}"
            )

            if metrics["reward_accuracy"] > best_accuracy:
                best_accuracy = metrics["reward_accuracy"]

                save_checkpoint(policy, best_dir)

                zip_checkpoint(
                    best_dir,
                    out / "best_checkpoint.zip"
                )

                print("Saved new best checkpoint.")

        for epoch in range(epochs):
            policy.train()
            accumulated = 0

            for batch_index, batch in enumerate(train_loader):

                # Policy forward: gradients enabled.
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    pc, pr = get_pair_logps(
                        policy, batch, "cuda:0"
                    )

                # Reference forward: frozen.
                with torch.no_grad():
                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        rc, rr = get_pair_logps(
                            reference, batch, "cuda:1"
                        )

                loss, accuracy, margin = dpo_loss_from_logps(
                    pc.float(),
                    pr.float(),
                    rc.float().to("cuda:0"),
                    rr.float().to("cuda:0"),
                    beta
                )

                if not torch.isfinite(loss).item():
                    raise RuntimeError(
                        f"Nonfinite loss at batch {batch_index}"
                    )

                scaler.scale(loss).backward()
                accumulated += 1

                last_batch = batch_index + 1 == len(train_loader)

                if accumulated < grad_accum and not last_batch:
                    continue

                scaler.unscale_(optimizer)

                # Average by the actual accumulation group size.
                for p in policy.parameters():
                    if p.grad is not None:
                        p.grad.div_(accumulated)

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    policy.parameters(),
                    max_norm=1.0,
                    error_if_nonfinite=False
                )

                old_scale = scaler.get_scale()

                scaler.step(optimizer)
                scaler.update()

                new_scale = scaler.get_scale()

                optimizer.zero_grad(set_to_none=True)
                accumulated = 0

                if new_scale < old_scale:
                    skipped += 1

                    print(
                        f"Skipped update: "
                        f"grad_norm={grad_norm.item():.4g}, "
                        f"scale={old_scale:g}->{new_scale:g}"
                    )

                    if steps == 0 and skipped >= 5:
                        raise RuntimeError(
                            "Five consecutive optimizer updates "
                            "were skipped. Stop and diagnose "
                            "nonfinite FP16 gradients."
                        )

                    continue

                steps += 1
                progress.update(1)

                progress.set_postfix(
                    loss=f"{loss.item():.4f}",
                    grad=f"{grad_norm.item():.3f}",
                    scale=f"{new_scale:g}"
                )

                if steps % eval_every == 0:
                    run_validation()

        if steps == 0:
            raise RuntimeError(
                "Training ended at Step 0. "
                "No successful optimizer update occurred."
            )

        run_validation()

        save_checkpoint(policy, final_dir)

        zip_checkpoint(
            final_dir,
            out / "final_checkpoint.zip"
        )

        summary = {
            "run": run_name,
            "successful_optimizer_updates": steps,
            "skipped_updates": skipped,
            "best_validation_reward_accuracy": best_accuracy,
            "train_pairs": len(train_loader.dataset),
            "validation_pairs": len(val_loader.dataset),
            "beta": beta,
            "learning_rate": lr,
            "gradient_accumulation": grad_accum,
            "epochs": epochs,
            "optimizer": "bitsandbytes AdamW8bit",
            "initial_checkpoint": str(SFT_BEST),
            "reference_checkpoint": str(SFT_BEST)
        }

        with open(out / "training_summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        print("\nTRAINING COMPLETED")
        print(json.dumps(summary, indent=2))

    finally:
        progress.close()

        del optimizer, scaler, policy, reference
        gc.collect()
        torch.cuda.empty_cache()

    return out

In [ ]:
test_loader = DataLoader(
    Subset(hh_train_loader.dataset, range(32)),
    batch_size=1,
    shuffle=False,
    collate_fn=dpo_collate
)

test_output = train_dpo(
    run_name="dpo_hh_memory_test",
    train_loader=test_loader,
    val_loader=hh_val_loader,
    beta=BETA,
    lr=LR,
    grad_accum=GRAD_ACCUM,
    epochs=1,
    eval_every=150
)


In [ ]:
hh_output = train_dpo(
    run_name="dpo_hh",
    train_loader=hh_train_loader,
    val_loader=hh_val_loader,
    beta=BETA,
    lr=LR,
    grad_accum=GRAD_ACCUM,
    epochs=EPOCHS,
    eval_every=EVAL_EVERY
)

In [ ]:
for filename in [
    "best/model.safetensors",
    "final/model.safetensors",
    "best_checkpoint.zip",
    "final_checkpoint.zip",
    "metrics.jsonl",
    "training_summary.json"
]:
    path = HH_DIR / filename
    assert path.exists(), f"Missing: {path}"
    print("OK:", filename)

with open(HH_DIR / "training_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
uf_output = train_dpo(
    run_name="dpo_ultrafeedback",
    train_loader=uf_train_loader,
    val_loader=uf_val_loader,
    beta=BETA,
    lr=LR,
    grad_accum=GRAD_ACCUM,
    epochs=EPOCHS,
    eval_every=EVAL_EVERY
)